# Verbesserte Automerkennung auf Bildern (PIL-Version)

Dieses Notebook implementiert eine verbesserte Automerkennung auf Bildern mit dem trainierten CNN-Modell. Es verwendet einen kombinierten Ansatz mit Selective Search für Region Proposals und Multi-Scale-Erkennung, um Autos in Bildern zuverlässiger zu lokalisieren und zu markieren. Diese Version verwendet PIL anstelle von OpenCV für die Bildverarbeitung, um Kompatibilität mit Python 3.11 zu gewährleisten.

## Einführung und theoretischer Hintergrund

Die Objekterkennung in Bildern ist eine komplexe Aufgabe, die über die einfache Klassifikation hinausgeht. Während die Klassifikation bestimmt, ob ein Bild ein bestimmtes Objekt enthält, lokalisiert die Objekterkennung zusätzlich die Position des Objekts im Bild durch Bounding Boxes.

Moderne Objekterkennungssysteme wie YOLO (You Only Look Once), SSD (Single Shot Detector) oder Faster R-CNN sind End-to-End-Lösungen, die direkt Bounding Boxes und Klassifikationen ausgeben. In diesem Notebook implementieren wir jedoch einen zweistufigen Ansatz, der die Grundprinzipien der Objekterkennung verdeutlicht:

1. **Region Proposal**: Identifizierung potenzieller Bereiche im Bild, die Objekte enthalten könnten
2. **Klassifikation**: Anwendung eines CNN-Klassifikators auf jeden vorgeschlagenen Bereich

Dieser Ansatz ähnelt dem R-CNN (Region-based Convolutional Neural Network) Algorithmus, der ein Meilenstein in der Entwicklung moderner Objekterkennungssysteme war.

Für die Region Proposals implementieren wir einen alternativen Algorithmus zu Selective Search, der auf Sliding Windows und Multi-Scale-Analyse basiert. Dieser Ansatz hat folgende Vorteile:

- **Flexibilität**: Anpassbar an verschiedene Objektgrößen und -formen
- **Einfachheit**: Leichter zu implementieren und zu verstehen als komplexere Algorithmen
- **Kompatibilität**: Funktioniert mit PIL anstelle von OpenCV, was die Portabilität verbessert

Nach der Generierung der Region Proposals und deren Klassifikation wenden wir Non-Maximum Suppression (NMS) an, um überlappende Bounding Boxes zu entfernen. NMS ist ein wichtiger Nachbearbeitungsschritt in der Objekterkennung, der die endgültigen Erkennungsergebnisse verbessert, indem er redundante Detektionen eliminiert.

Die Implementierung in diesem Notebook zeigt, wie man grundlegende Techniken der Objekterkennung kombinieren kann, um ein funktionierendes System zu erstellen, ohne auf spezialisierte Bibliotheken oder vortrainierte Objekterkennungsmodelle angewiesen zu sein.

## Überblick über die Schritte
- Laden des trainierten CNN-Modells
- Implementierung eines alternativen Algorithmus für Region Proposals
- Verbesserte Multi-Scale-Erkennung für verschiedene Objektgrößen
- Optimierte Non-Maximum Suppression zur Entfernung überlappender Bounding Boxes
- Anwendung auf Testbilder und Visualisierung der Ergebnisse

## Importieren der benötigten Bibliotheken

Für unsere verbesserte Automerkennung benötigen wir verschiedene Bibliotheken:

- **numpy**: Für effiziente numerische Operationen und Array-Manipulationen
- **matplotlib**: Für die Visualisierung der Erkennungsergebnisse
- **tensorflow**: Zum Laden und Verwenden unseres trainierten CNN-Modells
- **PIL (Python Imaging Library)**: Für die Bildverarbeitung anstelle von OpenCV
- **requests**: Zum Herunterladen von Testbildern aus dem Internet
- **skimage**: Für die Extraktion von HOG-Features (Histogram of Oriented Gradients)

Die Verwendung von PIL anstelle von OpenCV bietet bessere Kompatibilität mit verschiedenen Python-Versionen, insbesondere mit Python 3.11, und reduziert die Abhängigkeit von komplexen Bibliotheken.

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
import os
import requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
import time
from skimage.feature import hog
from skimage import exposure

## Vorbereitung der Verzeichnisse

Bevor wir mit der Implementierung beginnen, erstellen wir Verzeichnisse für die Speicherung der Modelle, Testbilder und Ergebnisse. Eine gute Organisation der Projektstruktur ist wichtig für die Nachvollziehbarkeit und Wiederverwendbarkeit des Codes.

Wir erstellen drei Verzeichnisse:
- `models_dir`: Für den Zugriff auf die trainierten Modelle
- `test_images_dir`: Für die Speicherung der Testbilder
- `results_dir`: Für die Speicherung der Erkennungsergebnisse

Die Funktion `os.makedirs()` mit dem Parameter `exist_ok=True` stellt sicher, dass kein Fehler auftritt, falls die Verzeichnisse bereits existieren.

In [ ]:
# Vorbereitung der Verzeichnisse
models_dir = '../models'
keras_models_dir = os.path.join(models_dir, 'keras')
test_images_dir = '../test_images'
results_dir = '../results'

os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

## Laden des trainierten CNN-Modells

Wir laden das in Notebook 2 trainierte CNN-Modell, das auf dem CIFAR-10-Datensatz trainiert wurde, um Autos zu erkennen. Dieses Modell wurde mit Keras und TensorFlow implementiert und als .h5-Datei gespeichert.

Das Modell erwartet Eingabebilder mit einer Größe von 32x32 Pixeln und 3 Farbkanälen (RGB) und gibt eine binäre Klassifikation aus (1 für Auto, 0 für Nicht-Auto). Wir werden dieses Modell verwenden, um die vorgeschlagenen Regionen in größeren Bildern zu klassifizieren.

Alternativ könnten wir auch das vortrainierte MobileNetV2-Modell aus Notebook 4 verwenden, das möglicherweise eine bessere Genauigkeit bietet, aber für dieses Beispiel verwenden wir das einfachere Keras-Modell.

In [ ]:
# Laden des trainierten CNN-Modells
model_path = os.path.join(keras_models_dir, 'car_classifier_model.h5')

try:
    model = load_model(model_path)
    print(f"Modell erfolgreich geladen von: {model_path}")
    model.summary()
except Exception as e:
    print(f"Fehler beim Laden des Modells: {e}")
    print("Versuche, das Modell aus einem anderen Verzeichnis zu laden...")
    
    # Alternative Pfade probieren
    alternative_paths = [
        os.path.join(models_dir, 'car_classifier_model.h5'),
        os.path.join(models_dir, 'pretrained', 'mobilenet_final_model.h5')
    ]
    
    for alt_path in alternative_paths:
        try:
            model = load_model(alt_path)
            print(f"Modell erfolgreich geladen von: {alt_path}")
            model.summary()
            break
        except:
            continue
    else:
        print("Konnte kein Modell laden. Bitte stellen Sie sicher, dass ein trainiertes Modell verfügbar ist.")

## Hilfsfunktionen für die Bildverarbeitung

Bevor wir mit der Implementierung des Objekterkennungsalgorithmus beginnen, definieren wir einige Hilfsfunktionen für die Bildverarbeitung. Diese Funktionen werden verwendet, um Bilder zu laden, zu verarbeiten und für die Klassifikation vorzubereiten.

Die wichtigsten Funktionen sind:
1. `load_image_from_url`: Lädt ein Bild von einer URL
2. `load_image_from_file`: Lädt ein Bild aus einer Datei
3. `preprocess_image`: Bereitet ein Bild für die Klassifikation vor

Diese Funktionen verwenden PIL für die Bildverarbeitung, was eine bessere Kompatibilität mit verschiedenen Python-Versionen bietet als OpenCV.

In [ ]:
def load_image_from_url(url):
    """
    Lädt ein Bild von einer URL.
    
    Parameter:
    - url: URL des Bildes
    
    Rückgabe:
    - image: PIL Image-Objekt
    """
    try:
        response = requests.get(url)
        image = Image.open(BytesIO(response.content))
        return image
    except Exception as e:
        print(f"Fehler beim Laden des Bildes von URL: {e}")
        return None

def load_image_from_file(file_path):
    """
    Lädt ein Bild aus einer Datei.
    
    Parameter:
    - file_path: Pfad zur Bilddatei
    
    Rückgabe:
    - image: PIL Image-Objekt
    """
    try:
        image = Image.open(file_path)
        return image
    except Exception as e:
        print(f"Fehler beim Laden des Bildes aus Datei: {e}")
        return None

def preprocess_image(image, target_size=(32, 32)):
    """
    Bereitet ein Bild für die Klassifikation vor.
    
    Parameter:
    - image: PIL Image-Objekt
    - target_size: Zielgröße für das Bild (Höhe, Breite)
    
    Rückgabe:
    - processed_image: Vorverarbeitetes Bild als NumPy-Array mit Form (1, Höhe, Breite, Kanäle)
    """
    # Konvertieren zu RGB, falls notwendig
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Skalieren auf die Zielgröße
    image = image.resize(target_size, Image.LANCZOS)
    
    # Konvertieren zu NumPy-Array und normalisieren
    img_array = np.array(image).astype('float32') / 255.0
    
    # Erweitern der Dimensionen für das Batch
    processed_image = np.expand_dims(img_array, axis=0)
    
    return processed_image

## Implementierung des Region Proposal Algorithmus

Der Region Proposal Algorithmus ist ein wichtiger Bestandteil unseres Objekterkennungssystems. Er identifiziert potenzielle Bereiche im Bild, die Objekte enthalten könnten, und reduziert so den Suchraum für den Klassifikator.

Anstelle von Selective Search, das in OpenCV implementiert ist, verwenden wir einen alternativen Ansatz, der auf Sliding Windows und Multi-Scale-Analyse basiert. Dieser Ansatz hat folgende Vorteile:

1. **Kompatibilität**: Funktioniert mit PIL anstelle von OpenCV
2. **Anpassbarkeit**: Leicht an verschiedene Objektgrößen und -formen anzupassen
3. **Effizienz**: Kann durch Parameter wie Schrittweite und Skalierungsfaktoren optimiert werden

Der Algorithmus funktioniert wie folgt:
1. Definiere verschiedene Fenstergrößen (z.B. 64x64, 96x96, 128x128)
2. Für jede Fenstergröße, schiebe das Fenster über das Bild mit einer bestimmten Schrittweite
3. Für jeden Fensterbereich, extrahiere die Region als potenziellen Objektkandidaten

Dieser Ansatz generiert eine große Anzahl von überlappenden Regionen, die später durch Non-Maximum Suppression gefiltert werden.

In [ ]:
def generate_region_proposals(image, window_sizes, step_size):
    """
    Generiert Region Proposals mit einem Sliding-Window-Ansatz.
    
    Parameter:
    - image: PIL Image-Objekt
    - window_sizes: Liste von Fenstergrößen (Höhe, Breite)
    - step_size: Schrittweite für das Sliding Window
    
    Rückgabe:
    - regions: Liste von Regionen als (x, y, w, h) Tupel
    """
    regions = []
    width, height = image.size
    
    for window_size in window_sizes:
        for y in range(0, height - window_size + 1, step_size):
            for x in range(0, width - window_size + 1, step_size):
                regions.append((x, y, window_size, window_size))
    
    return regions

def extract_hog_features(image_region):
    """
    Extrahiert HOG-Features aus einer Bildregion.
    
    Parameter:
    - image_region: Bildregion als NumPy-Array
    
    Rückgabe:
    - features: HOG-Features als NumPy-Array
    """
    # Konvertieren zu Graustufen
    if len(image_region.shape) == 3 and image_region.shape[2] == 3:
        gray = np.mean(image_region, axis=2)
    else:
        gray = image_region
    
    # HOG-Features extrahieren
    features, hog_image = hog(
        gray, 
        orientations=9, 
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), 
        visualize=True, 
        feature_vector=True
    )
    
    return features, hog_image

def filter_regions_by_hog(image, regions, threshold=0.1):
    """
    Filtert Regionen basierend auf HOG-Features.
    
    Parameter:
    - image: PIL Image-Objekt
    - regions: Liste von Regionen als (x, y, w, h) Tupel
    - threshold: Schwellenwert für die HOG-Feature-Magnitude
    
    Rückgabe:
    - filtered_regions: Gefilterte Liste von Regionen
    """
    filtered_regions = []
    image_array = np.array(image)
    
    for x, y, w, h in regions:
        # Region extrahieren
        region = image_array[y:y+h, x:x+w]
        
        # HOG-Features extrahieren
        try:
            features, _ = extract_hog_features(region)
            
            # Filtern basierend auf der Feature-Magnitude
            if np.mean(features) > threshold:
                filtered_regions.append((x, y, w, h))
        except Exception as e:
            # Ignorieren von Regionen, die Probleme verursachen
            continue
    
    return filtered_regions

## Implementierung der Objekterkennung

Mit dem Region Proposal Algorithmus und dem geladenen CNN-Modell können wir nun die Objekterkennung implementieren. Der Prozess umfasst folgende Schritte:

1. **Region Proposals generieren**: Identifizierung potenzieller Bereiche im Bild
2. **Regionen klassifizieren**: Anwendung des CNN-Modells auf jede Region
3. **Filterung der Ergebnisse**: Entfernung von Regionen mit niedriger Konfidenz
4. **Non-Maximum Suppression**: Entfernung überlappender Bounding Boxes

Die Non-Maximum Suppression (NMS) ist ein wichtiger Schritt, um redundante Detektionen zu eliminieren. Der Algorithmus funktioniert wie folgt:
1. Sortiere alle Bounding Boxes nach ihrer Konfidenz
2. Wähle die Box mit der höchsten Konfidenz und entferne sie aus der Liste
3. Berechne die IoU (Intersection over Union) zwischen dieser Box und allen verbleibenden Boxen
4. Entferne alle Boxen mit einer IoU über einem bestimmten Schwellenwert
5. Wiederhole die Schritte 2-4, bis keine Boxen mehr übrig sind

Dieser Prozess stellt sicher, dass wir für jedes Objekt nur eine Bounding Box behalten, nämlich diejenige mit der höchsten Konfidenz.

In [ ]:
def calculate_iou(box1, box2):
    """
    Berechnet die Intersection over Union (IoU) zwischen zwei Bounding Boxes.
    
    Parameter:
    - box1: Erste Box als (x, y, w, h) Tupel
    - box2: Zweite Box als (x, y, w, h) Tupel
    
    Rückgabe:
    - iou: Intersection over Union Wert
    """
    # Konvertieren zu (x1, y1, x2, y2) Format
    x1_1, y1_1, w1, h1 = box1
    x2_1, y2_1 = x1_1 + w1, y1_1 + h1
    
    x1_2, y1_2, w2, h2 = box2
    x2_2, y2_2 = x1_2 + w2, y1_2 + h2
    
    # Berechnen der Koordinaten des Schnittbereichs
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    # Berechnen der Fläche des Schnittbereichs
    if x2_i <= x1_i or y2_i <= y1_i:
        return 0.0  # Keine Überlappung
    
    intersection_area = (x2_i - x1_i) * (y2_i - y1_i)
    
    # Berechnen der Flächen der beiden Boxen
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    # Berechnen der Union-Fläche
    union_area = box1_area + box2_area - intersection_area
    
    # Berechnen der IoU
    iou = intersection_area / union_area
    
    return iou

def non_max_suppression(boxes, scores, iou_threshold=0.5):
    """
    Führt Non-Maximum Suppression auf Bounding Boxes durch.
    
    Parameter:
    - boxes: Liste von Bounding Boxes als (x, y, w, h) Tupel
    - scores: Liste von Konfidenzwerten für jede Box
    - iou_threshold: Schwellenwert für die IoU
    
    Rückgabe:
    - selected_boxes: Liste von ausgewählten Bounding Boxes
    - selected_scores: Liste von Konfidenzwerten für die ausgewählten Boxen
    """
    # Sortieren der Boxen nach Konfidenz (absteigend)
    indices = np.argsort(scores)[::-1]
    boxes = [boxes[i] for i in indices]
    scores = [scores[i] for i in indices]
    
    selected_boxes = []
    selected_scores = []
    
    while len(boxes) > 0:
        # Die Box mit der höchsten Konfidenz auswählen
        current_box = boxes[0]
        current_score = scores[0]
        
        selected_boxes.append(current_box)
        selected_scores.append(current_score)
        
        # Entfernen der ausgewählten Box
        boxes.pop(0)
        scores.pop(0)
        
        # Filtern der verbleibenden Boxen
        i = 0
        while i < len(boxes):
            iou = calculate_iou(current_box, boxes[i])
            if iou > iou_threshold:
                # Entfernen von Boxen mit hoher Überlappung
                boxes.pop(i)
                scores.pop(i)
            else:
                i += 1
    
    return selected_boxes, selected_scores

def detect_cars(image, model, confidence_threshold=0.7, iou_threshold=0.5):
    """
    Erkennt Autos in einem Bild mit dem trainierten CNN-Modell.
    
    Parameter:
    - image: PIL Image-Objekt
    - model: Trainiertes CNN-Modell
    - confidence_threshold: Schwellenwert für die Konfidenz
    - iou_threshold: Schwellenwert für die IoU bei Non-Maximum Suppression
    
    Rückgabe:
    - detections: Liste von Detektionen als (x, y, w, h, score) Tupel
    """
    # Definieren der Fenstergrößen für Multi-Scale-Erkennung
    window_sizes = [64, 96, 128, 160, 192]
    step_size = 32
    
    # Region Proposals generieren
    regions = generate_region_proposals(image, window_sizes, step_size)
    print(f"Generierte {len(regions)} Region Proposals")
    
    # Filtern der Regionen basierend auf HOG-Features
    filtered_regions = filter_regions_by_hog(image, regions)
    print(f"Nach HOG-Filterung verbleiben {len(filtered_regions)} Regionen")
    
    # Klassifizieren der Regionen
    boxes = []
    scores = []
    
    for i, (x, y, w, h) in enumerate(filtered_regions):
        # Region extrahieren und vorverarbeiten
        region = image.crop((x, y, x+w, y+h))
        processed_region = preprocess_image(region)
        
        # Klassifizieren der Region
        prediction = model.predict(processed_region, verbose=0)[0][0]
        
        # Regionen mit hoher Konfidenz speichern
        if prediction > confidence_threshold:
            boxes.append((x, y, w, h))
            scores.append(float(prediction))
    
    print(f"Nach Klassifikation verbleiben {len(boxes)} Regionen mit Konfidenz > {confidence_threshold}")
    
    # Non-Maximum Suppression anwenden
    if len(boxes) > 0:
        selected_boxes, selected_scores = non_max_suppression(boxes, scores, iou_threshold)
        print(f"Nach Non-Maximum Suppression verbleiben {len(selected_boxes)} Detektionen")
        
        # Detektionen als (x, y, w, h, score) Tupel zurückgeben
        detections = [(box[0], box[1], box[2], box[3], score) for box, score in zip(selected_boxes, selected_scores)]
        return detections
    else:
        print("Keine Autos erkannt")
        return []

## Visualisierung der Erkennungsergebnisse

Nach der Implementierung des Objekterkennungsalgorithmus benötigen wir eine Funktion zur Visualisierung der Ergebnisse. Diese Funktion zeichnet Bounding Boxes um die erkannten Objekte und zeigt die Konfidenzwerte an.

Die Visualisierung ist ein wichtiger Schritt, um die Leistung des Algorithmus zu bewerten und mögliche Probleme zu identifizieren. Sie hilft uns auch, die Ergebnisse besser zu verstehen und zu kommunizieren.

In [ ]:
def visualize_detections(image, detections, output_path=None):
    """
    Visualisiert die Erkennungsergebnisse.
    
    Parameter:
    - image: PIL Image-Objekt
    - detections: Liste von Detektionen als (x, y, w, h, score) Tupel
    - output_path: Pfad zum Speichern des Ergebnisbildes (optional)
    
    Rückgabe:
    - result_image: PIL Image-Objekt mit gezeichneten Bounding Boxes
    """
    # Kopie des Bildes erstellen
    result_image = image.copy()
    draw = ImageDraw.Draw(result_image)
    
    # Versuchen, eine Schriftart zu laden
    try:
        font = ImageFont.truetype("arial.ttf", 16)
    except IOError:
        font = ImageFont.load_default()
    
    # Bounding Boxes zeichnen
    for x, y, w, h, score in detections:
        # Rechteck zeichnen
        draw.rectangle([x, y, x+w, y+h], outline="red", width=3)
        
        # Text mit Konfidenz zeichnen
        text = f"Auto: {score:.2f}"
        text_width, text_height = draw.textsize(text, font=font) if hasattr(draw, 'textsize') else (100, 20)
        draw.rectangle([x, y-text_height-4, x+text_width+4, y], fill="red")
        draw.text((x+2, y-text_height-2), text, fill="white", font=font)
    
    # Bild speichern, falls ein Ausgabepfad angegeben wurde
    if output_path:
        result_image.save(output_path)
    
    return result_image

## Testbilder herunterladen

Bevor wir unseren Objekterkennungsalgorithmus testen können, benötigen wir einige Testbilder. Wir laden Bilder von Autos aus dem Internet herunter und speichern sie im Testbilder-Verzeichnis.

Die Bilder sollten verschiedene Szenarien abdecken, um die Robustheit des Algorithmus zu testen:
- Autos in verschiedenen Perspektiven (Vorderansicht, Seitenansicht, Rückansicht)
- Autos in verschiedenen Umgebungen (Stadt, Landstraße, Parkplatz)
- Bilder mit mehreren Autos
- Bilder mit anderen Objekten neben Autos

Diese Vielfalt an Testbildern hilft uns, die Stärken und Schwächen unseres Algorithmus zu identifizieren.

In [ ]:
# URLs für Testbilder
test_image_urls = [
    "https://cdn.pixabay.com/photo/2015/05/28/23/12/auto-788747_1280.jpg",  # Einzelnes Auto
    "https://cdn.pixabay.com/photo/2016/11/18/12/51/automobile-1834274_1280.jpg",  # Mehrere Autos
    "https://cdn.pixabay.com/photo/2017/01/28/17/43/car-2016158_1280.jpg",  # Auto in Stadtumgebung
    "https://cdn.pixabay.com/photo/2016/04/01/12/16/car-1300629_1280.png",  # Cartoon-Auto
    "https://cdn.pixabay.com/photo/2014/09/07/22/34/car-race-438467_1280.jpg"  # Rennwagen
]

# Testbilder herunterladen
test_images = []
test_image_paths = []

for i, url in enumerate(test_image_urls):
    try:
        # Bild herunterladen
        image = load_image_from_url(url)
        
        if image:
            # Bild speichern
            image_path = os.path.join(test_images_dir, f"test_image_{i+1}.jpg")
            image.save(image_path)
            
            # Bild und Pfad speichern
            test_images.append(image)
            test_image_paths.append(image_path)
            
            print(f"Bild {i+1} erfolgreich heruntergeladen und gespeichert unter: {image_path}")
    except Exception as e:
        print(f"Fehler beim Herunterladen von Bild {i+1}: {e}")

print(f"Insgesamt {len(test_images)} Testbilder heruntergeladen")

# Testbilder anzeigen
plt.figure(figsize=(15, 10))
for i, image in enumerate(test_images):
    plt.subplot(2, 3, i+1)
    plt.imshow(image)
    plt.title(f"Testbild {i+1}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## Anwendung der Objekterkennung auf Testbilder

Jetzt können wir unseren Objekterkennungsalgorithmus auf die heruntergeladenen Testbilder anwenden. Wir werden die Ergebnisse visualisieren und im Ergebnisverzeichnis speichern.

Dieser Schritt ermöglicht es uns, die Leistung des Algorithmus zu bewerten und mögliche Verbesserungen zu identifizieren. Wir können verschiedene Parameter wie den Konfidenz-Schwellenwert oder den IoU-Schwellenwert anpassen, um die Ergebnisse zu optimieren.

In [ ]:
# Objekterkennung auf Testbildern anwenden
for i, (image, image_path) in enumerate(zip(test_images, test_image_paths)):
    print(f"\nVerarbeite Testbild {i+1}...")
    
    # Autos erkennen
    start_time = time.time()
    detections = detect_cars(image, model, confidence_threshold=0.7, iou_threshold=0.5)
    elapsed_time = time.time() - start_time
    
    print(f"Erkennungszeit: {elapsed_time:.2f} Sekunden")
    print(f"Erkannte {len(detections)} Autos")
    
    # Ergebnisse visualisieren
    output_path = os.path.join(results_dir, f"result_image_{i+1}.jpg")
    result_image = visualize_detections(image, detections, output_path)
    
    # Ergebnisse anzeigen
    plt.figure(figsize=(10, 8))
    plt.imshow(result_image)
    plt.title(f"Testbild {i+1} - {len(detections)} Autos erkannt")
    plt.axis('off')
    plt.show()

## Optimierung der Parameter

Die Leistung unseres Objekterkennungsalgorithmus hängt von verschiedenen Parametern ab, die wir optimieren können. Die wichtigsten Parameter sind:

1. **Konfidenz-Schwellenwert**: Bestimmt, ab welcher Konfidenz eine Region als Auto klassifiziert wird
2. **IoU-Schwellenwert**: Bestimmt, ab welcher Überlappung Bounding Boxes als redundant betrachtet werden
3. **Fenstergrößen**: Bestimmen die Größen der Sliding Windows für die Region Proposals
4. **Schrittweite**: Bestimmt, wie weit das Sliding Window bei jedem Schritt bewegt wird

Wir können diese Parameter anpassen, um die Genauigkeit und Effizienz des Algorithmus zu verbessern. Eine niedrigere Konfidenz führt zu mehr Detektionen, aber auch zu mehr Falsch-Positiven, während ein höherer IoU-Schwellenwert zu mehr überlappenden Boxen führt.

In [ ]:
# Optimierung der Parameter
def optimize_parameters(image, model):
    """
    Testet verschiedene Parameter für die Objekterkennung.
    
    Parameter:
    - image: PIL Image-Objekt
    - model: Trainiertes CNN-Modell
    """
    # Verschiedene Konfidenz-Schwellenwerte testen
    confidence_thresholds = [0.5, 0.7, 0.9]
    
    plt.figure(figsize=(15, 10))
    for i, conf_threshold in enumerate(confidence_thresholds):
        print(f"\nTeste Konfidenz-Schwellenwert: {conf_threshold}")
        
        # Autos erkennen
        detections = detect_cars(image, model, confidence_threshold=conf_threshold, iou_threshold=0.5)
        
        # Ergebnisse visualisieren
        result_image = visualize_detections(image, detections)
        
        # Ergebnisse anzeigen
        plt.subplot(2, 2, i+1)
        plt.imshow(result_image)
        plt.title(f"Konfidenz > {conf_threshold} - {len(detections)} Autos erkannt")
        plt.axis('off')
    
    # Verschiedene IoU-Schwellenwerte testen
    iou_threshold = 0.3
    detections = detect_cars(image, model, confidence_threshold=0.7, iou_threshold=iou_threshold)
    result_image = visualize_detections(image, detections)
    
    plt.subplot(2, 2, 4)
    plt.imshow(result_image)
    plt.title(f"IoU > {iou_threshold} - {len(detections)} Autos erkannt")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Parameter für ein Testbild optimieren
if len(test_images) > 0:
    optimize_parameters(test_images[1], model)  # Zweites Bild verwenden (mit mehreren Autos)

## Zusammenfassung und Ausblick

In diesem Notebook haben wir eine verbesserte Automerkennung auf Bildern implementiert, die einen kombinierten Ansatz mit Region Proposals und CNN-Klassifikation verwendet. Hier sind die wichtigsten Punkte:

1. **Region Proposal Algorithmus**: Wir haben einen alternativen Algorithmus zu Selective Search implementiert, der auf Sliding Windows und Multi-Scale-Analyse basiert und mit PIL anstelle von OpenCV funktioniert.

2. **HOG-Feature-Filterung**: Wir haben HOG-Features verwendet, um die Anzahl der Region Proposals zu reduzieren und die Effizienz zu verbessern.

3. **CNN-Klassifikation**: Wir haben ein trainiertes CNN-Modell verwendet, um die vorgeschlagenen Regionen zu klassifizieren und Autos zu erkennen.

4. **Non-Maximum Suppression**: Wir haben NMS implementiert, um überlappende Bounding Boxes zu entfernen und die endgültigen Erkennungsergebnisse zu verbessern.

5. **Parameteroptimierung**: Wir haben verschiedene Parameter wie den Konfidenz-Schwellenwert und den IoU-Schwellenwert getestet, um die Leistung des Algorithmus zu optimieren.

Obwohl unser Ansatz funktioniert, gibt es noch Raum für Verbesserungen:

- **Effizienz**: Der Algorithmus könnte durch Techniken wie Bildpyramiden oder effizientere Feature-Extraktion beschleunigt werden.
- **Genauigkeit**: Die Genauigkeit könnte durch ein besseres Modell oder durch Feintuning auf einem spezifischen Datensatz verbessert werden.
- **Robustheit**: Der Algorithmus könnte robuster gegenüber verschiedenen Lichtbedingungen, Perspektiven und Verdeckungen gemacht werden.

Für Produktionsanwendungen würden wir moderne End-to-End-Objekterkennungssysteme wie YOLO, SSD oder Faster R-CNN empfehlen, die effizienter und genauer sind. Unser Ansatz dient jedoch als gute Einführung in die Grundprinzipien der Objekterkennung und zeigt, wie man verschiedene Techniken kombinieren kann, um ein funktionierendes System zu erstellen.

Im nächsten Notebook werden wir einen Bonus-Anwendungsfall betrachten: die Erkennung von Personen mit ähnlichen Techniken.